# Maximum Gram approximation error for five sample sizes

For each $n\in\{200,400,600,800,1000\}$, use $M=200$ independent samples from $N(0,1)$, $\Omega=450$, and $N=50$ geometrically spaced bandwidths in $[0.01,2]$. Plot five curves:

$$E_K^{\max}(n)=\max_{1\le m\le200}\max_{1\le r,s\le50}\left|G_{rs}^{(m,n)}-G_{rs}^{(m,n),450,K}\right|.$$

Each curve takes the maximum over both the 200 realizations and every bandwidth pair, including diagonal entries. It measures total error, including truncation and quadrature. The dashed horizontal line is the $10^{-8}$ target; the dotted vertical line is the current $K=1536$.

Within realization $m$, the five sample sizes are nested prefixes of one length-1000 sequence. Different realizations use independent random seeds `20261024 + m`, with zero-based indexing. Thus, the 200 samples are independent within each curve, while the five curves share observations to make sample-size comparisons consistent. The $n=200$ observations exactly match [the previous repeated-sample experiment](g_rs_error_vs_k_repeated_samples.ipynb).

The analytic reference is the Gaussian formula in [density.py](../kde_tools/density.py), from the thesis's Equation `eq:beta2_exact_formula02`. The approximation is the same trapezoidal frequency integral as [the production implementation](../kde_tools/fourier/similarities.py), with $K$ intervals and $K+1$ nodes.

The calculations below share work across nested prefixes. Our algebraic rearrangement of the exact Gaussian sum uses the $n$ diagonal terms plus twice the lower-triangular sum. Fourier phase sums are accumulated in blocks of 200 observations. Before the full sweep, both rearrangements are compared with the existing implementations for all five sample sizes. Per-realization checkpoints allow the notebook to resume without mixing settings. Results are empirical, rather than a uniform accuracy guarantee.

Every displayed figure is automatically saved as PDF under `../outputs/pdfs/g_rs_error_vs_k_sample_sizes/`. Settings, actual samples, reference matrices, individual errors, summary CSV, and the PNG preview are also saved under `../outputs/`.

In [ ]:
from pathlib import Path
import json
import hashlib
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from time import perf_counter
import sys

code_directory = next(
    (candidate.resolve()
     for ancestor in (Path.cwd(), *Path.cwd().parents)
     for candidate in (ancestor, ancestor / 'Code' / 'Code_kde')
     if (candidate / 'kde_tools' / 'pdf_export.py').is_file()),
    None,
)
if code_directory is None:
    raise FileNotFoundError('Run this notebook from inside the MASTERARBEIT project.')
if str(code_directory) not in sys.path:
    sys.path.insert(0, str(code_directory))

import matplotlib.pyplot as plt
import numpy as np
from threadpoolctl import threadpool_limits

from kde_tools.density import exact_gaussian_kde_l2_gram
from kde_tools.fourier import (
    FOURIER_L2_METHOD_VERSION,
    gaussian_kde_l2_gram_fourier,
)
from kde_tools.pdf_export import enable_automatic_pdf_saving

notebook_name = 'g_rs_error_vs_k_sample_sizes'
pdf_output_directory = enable_automatic_pdf_saving(notebook_name)

In [ ]:
sample_sizes = np.arange(200, 1000 + 1, 200, dtype=int)
realizations = 200
seed = 20261024
N = 50
h_min, h_max = 0.01, 2.0
frequency_cutoff = 450.0
K_candidates = np.arange(64, 2304 + 1, 16, dtype=int)
highlight_K = 1536
absolute_tolerance = 1e-8
workers = min(4, os.cpu_count() or 1)
block_size = int(sample_sizes[0])
max_n = int(sample_sizes[-1])

samples = np.stack([
    np.random.default_rng(seed + m).normal(size=max_n)
    for m in range(realizations)
])
bandwidths = np.geomspace(h_min, h_max, N)
settings = {
    'sample_distribution': 'standard_normal', 'realizations_per_size': realizations,
    'sample_sizes': sample_sizes.tolist(), 'seed': seed,
    'seed_rule': 'seed + zero_based_sample_index; nested prefixes across n',
    'sample_sha256': hashlib.sha256(samples.tobytes()).hexdigest(),
    'N': N, 'h_min': h_min, 'h_max': h_max, 'bandwidth_grid': 'geometric',
    'frequency_cutoff': frequency_cutoff, 'K_candidates': K_candidates.tolist(),
    'highlight_K': highlight_K, 'absolute_tolerance': absolute_tolerance,
    'K_convention': 'trapezoidal intervals; frequency_points = K + 1',
    'curve_definition': 'max over realizations and all r,s of abs(G_exact - G_approx)',
    'reference': 'analytic_gaussian_pair_sum_float64',
    'method_version': FOURIER_L2_METHOD_VERSION,
    'experiment_engine': 'nested_prefix_v1', 'numpy_version': np.__version__,
}
configuration_id = hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()[:12]
data_directory = code_directory / 'outputs' / 'data' / notebook_name / configuration_id
checkpoint_directory = data_directory / 'realizations'
image_directory = code_directory / 'outputs' / 'images' / notebook_name
checkpoint_directory.mkdir(parents=True, exist_ok=True)
image_directory.mkdir(parents=True, exist_ok=True)
np.save(data_directory / 'samples.npy', samples)
np.save(data_directory / 'bandwidths.npy', bandwidths)
(data_directory / 'settings.json').write_text(json.dumps(settings, indent=2) + '\n')

# Templates depend only on K and the bandwidths, so all realizations share them.
frequency_templates = []
for K in K_candidates:
    omega = np.linspace(0.0, frequency_cutoff, int(K) + 1)
    weights = np.full(len(omega), (omega[1] - omega[0]) / np.pi)
    weights[[0, -1]] *= 0.5
    gaussian_features = np.exp(-0.5 * (bandwidths[:, None] * omega[None, :])**2)
    frequency_templates.append((omega, np.sqrt(weights), gaussian_features))

# Lower-triangular pairs are ordered by their larger observation index.
pair_i, pair_j = np.tril_indices(max_n, k=-1)
pair_counts = sample_sizes * (sample_sizes - 1) // 2
pair_group_starts = np.r_[0, pair_counts[:-1]]
print(f'Configuration {configuration_id}: {realizations} realizations per n={sample_sizes.tolist()}.')
print(f'Omega={frequency_cutoff:g}, N={N}, {len(K_candidates)} K values; up to {workers} calculation threads.')

In [ ]:
def exact_grams_for_prefixes(current_sample):
    negative_squared_differences = -(current_sample[pair_i] - current_sample[pair_j])**2
    workspace = np.empty_like(negative_squared_differences)
    references = np.empty((len(sample_sizes), N, N))
    for r, h_r in enumerate(bandwidths):
        for s in range(r, N):
            variance = h_r**2 + bandwidths[s]**2
            np.divide(negative_squared_differences, 2.0 * variance, out=workspace)
            np.exp(workspace, out=workspace)
            pair_sums = np.cumsum(np.add.reduceat(workspace, pair_group_starts))
            values = (sample_sizes + 2.0 * pair_sums) / (
                sample_sizes**2 * np.sqrt(2.0 * np.pi * variance)
            )
            references[:, r, s] = values
            references[:, s, r] = values
    return references

def approximate_grams_for_prefixes(current_sample, template):
    omega, root_weights, gaussian_features = template
    phases = np.empty((len(omega), max_n), dtype=np.complex128)
    np.multiply(omega[:, None], current_sample[None, :], out=phases)
    phases *= -1j
    np.exp(phases, out=phases)
    block_sums = phases.reshape(len(omega), len(sample_sizes), block_size).sum(axis=2)
    empirical_cf = np.cumsum(block_sums, axis=1) / sample_sizes[None, :]
    grams = np.empty((len(sample_sizes), N, N))
    for size_index in range(len(sample_sizes)):
        features = gaussian_features * (
            root_weights * np.abs(empirical_cf[:, size_index])
        )[None, :]
        grams[size_index] = features @ features.T
    return grams

def calculate_one_realization(sample_index):
    current_sample = samples[sample_index]
    references = exact_grams_for_prefixes(current_sample)
    local_errors = np.empty((len(sample_sizes), len(K_candidates)))
    local_worst = np.empty((len(sample_sizes), len(K_candidates), 2), dtype=int)
    for k_index, template in enumerate(frequency_templates):
        approximations = approximate_grams_for_prefixes(current_sample, template)
        differences = np.abs(approximations - references)
        flat_indices = differences.reshape(len(sample_sizes), -1).argmax(axis=1)
        r, s = np.unravel_index(flat_indices, (N, N))
        local_errors[:, k_index] = differences[np.arange(len(sample_sizes)), r, s]
        local_worst[:, k_index, 0] = r
        local_worst[:, k_index, 1] = s
    return sample_index, references, local_errors, local_worst

In [ ]:
# Validate the shared-prefix calculations against the existing implementations.
validation_start = perf_counter()
check_K = [64, 1024, highlight_K, int(K_candidates[-1])]
reference_discrepancies = []
approximation_discrepancies = []
with threadpool_limits(limits=1):
    prefix_references = exact_grams_for_prefixes(samples[0])
    for size_index, n in enumerate(sample_sizes):
        original_reference = exact_gaussian_kde_l2_gram(samples[0, :n], bandwidths)
        discrepancy = float(np.max(np.abs(original_reference - prefix_references[size_index])))
        reference_discrepancies.append(discrepancy)
        np.testing.assert_allclose(prefix_references[size_index], original_reference, rtol=0, atol=2e-14)
        print(f'Validated analytic reference at n={n}: discrepancy {discrepancy:.3e}.', flush=True)
    for K in check_K:
        k_index = int(np.flatnonzero(K_candidates == K)[0])
        shared_approximations = approximate_grams_for_prefixes(samples[0], frequency_templates[k_index])
        for size_index, n in enumerate(sample_sizes):
            original_approximation = gaussian_kde_l2_gram_fourier(
                samples[0, :n], bandwidths,
                frequency_cutoff=frequency_cutoff, frequency_points=K + 1,
            )
            discrepancy = float(np.max(np.abs(original_approximation - shared_approximations[size_index])))
            approximation_discrepancies.append(discrepancy)
            np.testing.assert_allclose(shared_approximations[size_index], original_approximation, rtol=0, atol=2e-14)
validation = {
    'sample_index': 0, 'sample_sizes': sample_sizes.tolist(), 'checked_K': check_K,
    'max_reference_discrepancy': max(reference_discrepancies),
    'max_approximation_discrepancy': max(approximation_discrepancies),
    'comparison_absolute_tolerance': 2e-14,
    'elapsed_seconds': perf_counter() - validation_start,
}
(data_directory / 'implementation_validation.json').write_text(json.dumps(validation, indent=2) + '\n')
print(f'All implementation comparisons passed in {validation["elapsed_seconds"]:.1f} s.', flush=True)

In [ ]:
per_sample_errors = np.empty((realizations, len(sample_sizes), len(K_candidates)))
worst_indices = np.empty((realizations, len(sample_sizes), len(K_candidates), 2), dtype=int)
exact_grams = np.empty((realizations, len(sample_sizes), N, N))
missing = []
for m in range(realizations):
    checkpoint_path = checkpoint_directory / f'{m:03d}.npz'
    if checkpoint_path.exists():
        with np.load(checkpoint_path) as checkpoint:
            if (str(checkpoint['configuration_id']) != configuration_id
                    or not np.array_equal(checkpoint['sample'], samples[m])):
                raise ValueError(f'Checkpoint inputs differ: {checkpoint_path}')
            per_sample_errors[m] = checkpoint['errors']
            worst_indices[m] = checkpoint['worst_indices']
            exact_grams[m] = checkpoint['references']
    else:
        missing.append(m)

start = perf_counter()
completed = realizations - len(missing)
print(f'Reused {completed} realization checkpoints; computing {len(missing)}.', flush=True)
with threadpool_limits(limits=1), ThreadPoolExecutor(max_workers=workers) as executor:
    futures = [executor.submit(calculate_one_realization, m) for m in missing]
    for future in as_completed(futures):
        m, references, local_errors, local_worst = future.result()
        exact_grams[m] = references
        per_sample_errors[m] = local_errors
        worst_indices[m] = local_worst
        checkpoint_path = checkpoint_directory / f'{m:03d}.npz'
        temporary_path = checkpoint_directory / f'{m:03d}.pending.npz'
        np.savez_compressed(
            temporary_path, configuration_id=configuration_id, sample=samples[m],
            references=references, errors=local_errors, worst_indices=local_worst,
        )
        temporary_path.replace(checkpoint_path)
        completed += 1
        if completed % 5 == 0 or completed == realizations:
            print(f'Completed {completed}/{realizations} realizations, all five sizes ({perf_counter() - start:.1f} s).', flush=True)
elapsed_seconds = perf_counter() - start
np.save(data_directory / 'exact_grams.npy', exact_grams)
np.savez_compressed(
    data_directory / 'per_sample_results.npz',
    K=K_candidates, sample_sizes=sample_sizes,
    errors=per_sample_errors, worst_indices=worst_indices,
)

In [ ]:
maximum_errors = per_sample_errors.max(axis=0)
worst_sample_indices = per_sample_errors.argmax(axis=0)
exceeding_counts = (per_sample_errors > absolute_tolerance).sum(axis=0)
highlight_index = int(np.flatnonzero(K_candidates == highlight_K)[0])
summary_rows = []
report_rows = []
for size_index, n in enumerate(sample_sizes):
    for k_index, K in enumerate(K_candidates):
        summary_rows.append([
            n, K, maximum_errors[size_index, k_index],
            worst_sample_indices[size_index, k_index], exceeding_counts[size_index, k_index],
        ])
    passing = np.flatnonzero(maximum_errors[size_index] <= absolute_tolerance)
    row = {
        'n': int(n), 'realizations': realizations,
        'first_tested_K_max_below_tolerance': int(K_candidates[passing[0]]) if len(passing) else None,
        'max_error_at_highlight_K': float(maximum_errors[size_index, highlight_index]),
        'samples_exceeding_tolerance_at_highlight_K': int(exceeding_counts[size_index, highlight_index]),
        'max_error_at_largest_K': float(maximum_errors[size_index, -1]),
    }
    report_rows.append(row)
    print(f'n={n}: first tested K below target={row["first_tested_K_max_below_tolerance"]}; max E_1536={row["max_error_at_highlight_K"]:.8e}.')
summary_table = np.asarray(summary_rows)
np.savetxt(
    data_directory / 'g_rs_error_vs_k_sample_sizes.csv', summary_table,
    delimiter=',', comments='',
    header='n,K,max_E_K,worst_sample_zero_based,samples_exceeding_tolerance',
    fmt=['%d', '%d', '%.17g', '%d', '%d'],
)
report = {
    'configuration_id': configuration_id, 'frequency_cutoff': frequency_cutoff,
    'N': N, 'realizations_per_size': realizations,
    'absolute_tolerance': absolute_tolerance, 'highlight_K': highlight_K,
    'largest_tested_K': int(K_candidates[-1]), 'sample_sizes': report_rows,
    'calculation_seconds_this_run': elapsed_seconds, 'workers': workers,
    'empirical_scope': 'maximum over 200 realizations and all 50-by-50 Gram entries for each n',
}
(data_directory / 'report.json').write_text(json.dumps(report, indent=2) + '\n')

In [ ]:
with plt.rc_context({
    'font.family': 'DejaVu Sans', 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False, 'pdf.fonttype': 42,
}):
    fig, ax = plt.subplots(figsize=(8.4, 5.0), layout='constrained')
    colors = ['#2266a5', '#d55e00', '#009e73', '#cc79a7', '#e69f00']
    styles = ['-', '--', '-.', ':', '-']
    for size_index, n in enumerate(sample_sizes):
        ax.plot(K_candidates, maximum_errors[size_index], color=colors[size_index],
                linestyle=styles[size_index], linewidth=2.0, label=rf'$n={n}$')
    ax.axhline(absolute_tolerance, color='#333333', linestyle='--', linewidth=1.2,
               label=r'Target $10^{-8}$')
    ax.axvline(highlight_K, color='#777777', linestyle=':', linewidth=1.1,
               label=rf'Current $K={highlight_K}$')
    ax.set_yscale('log')
    ax.set_xlim(0, K_candidates[-1])
    ax.set_xticks([0, 512, 1024, 1536, 2048])
    ax.set_xlabel(r'Number of trapezoidal intervals $K$')
    ax.set_ylabel(r'Maximum absolute Gram error over 200 samples')
    ax.set_title(
        rf'Maximum error for five sample sizes at $\Omega=450$' + '\n'
        + rf'$200$ samples per size; $N={N},\ h\in[{h_min:g},{h_max:g}]$; geometric grid',
        fontsize=12, pad=12,
    )
    ax.grid(axis='y', which='major', color='#e5e5e5', linewidth=0.7)
    ax.set_axisbelow(True)
    ax.legend(loc='upper right', frameon=False, fontsize=10)
    png_path = image_directory / 'g_rs_error_vs_k_five_sample_sizes.png'
    fig.savefig(png_path, dpi=200, bbox_inches='tight')
    plt.show()  # Also saves the PDF through kde_tools.pdf_export.